In [7]:
import pandas as pd
df = pd.read_csv(r"..\..\dataset\dataset_module_one_DM2.csv")

In [8]:
df["sii"].value_counts()

sii
0.0    5788
1.0    1573
2.0     940
3.0      83
Name: count, dtype: int64

## Preparazione dei Dati

In [9]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.utils import to_categorical
from imblearn.over_sampling import SMOTE

# Extract features and target
X = df.drop('sii', axis=1).values
y = df['sii'].values

# Train/Val/Test split (60-20-20)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)

target_counts = {
    1: 2000, # Moderate boost
    2: 2000, # Moderate boost
    3: 1500  # Targeted inflation as requested
}
smote = SMOTE(sampling_strategy=target_counts, random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# Feature scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_resampled)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# One-Hot Encoding for Categorical Cross-Entropy
y_train_ohe = to_categorical(y_train_resampled, num_classes=4)
y_val_ohe = to_categorical(y_val, num_classes=4)
y_test_ohe = to_categorical(y_test, num_classes=4)

## Costruzione del Modello e Prevenzione dell'Overfitting

In [11]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.metrics import F1Score, AUC
import keras_tuner as kt

def build_model(hp):
    model = Sequential()
    
    # Input & Hidden Layers tuning
    for i in range(hp.Int('num_layers', 1, 3)):
        model.add(Dense(
            units=hp.Int(f'units_{i}', min_value=32, max_value=128, step=32),
            activation='relu',
            kernel_regularizer=l2(0.001), # L2 Regularization
            input_shape=(X_train.shape[1],) if i == 0 else None
        ))
        # Dropout layer
        model.add(Dropout(hp.Float('dropout', 0.2, 0.5, step=0.1)))
    
    # Output Layer (4 nodes, Softmax)
    model.add(Dense(4, activation='softmax'))
    
    # Optimizer tuning (Adam vs RMSprop)
    optimizer = hp.Choice('optimizer', ['adam', 'rmsprop'])
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', 
                  metrics=[
                      AUC(name='auc'), F1Score(average='macro', name='macro_f1')
                    ])
    
    return model

## Hyper-parameter Tuning

In [13]:
from tensorflow.keras.callbacks import EarlyStopping

# Subclass RandomSearch to tune batch_size
class BatchTuner(kt.RandomSearch):
    def run_trial(self, trial, *args, **kwargs):
        kwargs['batch_size'] = trial.hyperparameters.Choice('batch_size', [32, 64, 128])
        return super().run_trial(trial, *args, **kwargs)

tuner = BatchTuner(
    build_model,
    objective=kt.Objective('val_macro_f1', direction='max'),
    max_trials=10,
    directory='tuning_logs',
    project_name='mlp_sii_2.0'
)

# Early Stopping callback
early_stopping = EarlyStopping(monitor='val_macro_f1', patience=10, restore_best_weights=True)

# Execute search
tuner.search(
    X_train, y_train_ohe,
    epochs=100,
    validation_data=(X_val, y_val_ohe),
    callbacks=[early_stopping],
    verbose=1,
)

best_model = tuner.get_best_models(num_models=1)[0]
best_model

Trial 10 Complete [00h 00m 08s]
val_macro_f1: 0.35473954677581787

Best val_macro_f1 So Far: 0.3613031506538391
Total elapsed time: 00h 04m 06s


c:\Users\abish\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\saving\saving_lib.py:798: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 8 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


<Sequential name=sequential, built=True>

## Test set e ROC

In [14]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, roc_curve, auc

# Predictions
y_pred_probs = best_model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

# Classification Report
print("Classification Report:\n")
print(classification_report(y_test, y_pred, digits=3))

# Multi-class ROC Curve (One-vs-Rest)
fpr, tpr, roc_auc = {}, {}, {}
for i in range(4):
    fpr[i], tpr[i], _ = roc_curve(y_test_ohe[:, i], y_pred_probs[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Plotting
plt.figure(figsize=(8, 6))
colors = ['blue', 'red', 'green', 'orange']

for i, color in zip(range(4), colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=2, label=f'Class {i} (AUC = {roc_auc[i]:.2f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2) # Diagonal reference
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Multi-class ROC Curve (One-vs-Rest)')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()

53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Classification Report:

              precision    recall  f1-score   support

         0.0      0.766     0.833     0.798      1150
         1.0      0.347     0.155     0.215       328
         2.0      0.241     0.367     0.291       177
         3.0      0.100     0.045     0.062        22

    accuracy                          0.641      1677
   macro avg      0.364     0.350     0.342      1677
weighted avg      0.620     0.641     0.621      1677



<Figure size 800x600 with 1 Axes>

# ==========================================
# SEZIONE XAI: EXPLAINABLE AI
# ==========================================

In [17]:
# Import necessari per XAI
from sklearn.tree import DecisionTreeClassifier, plot_tree, _tree
import matplotlib.pyplot as plt
import numpy as np
import shap
import warnings
warnings.filterwarnings('ignore')

# Assicuriamoci che i nomi delle feature siano disponibili (se non lo sono, li generiamo)
# Nel tuo codice originale "df" è il dataframe, quindi recuperiamo i nomi originali
feature_names = df.drop('sii', axis=1).columns.tolist()
class_names = ['0 (None)', '1 (Mild)', '2 (Moderate)', '3 (Severe)']

In [18]:
# ==============================================================================
# [1. TREPAN Global / Surrogato Globale]
# Addestriamo un Albero Decisionale per approssimare il comportamento globale 
# della Rete Neurale (Black Box).
# ==============================================================================
print("\n--- 1. SPIEGAZIONE GLOBALE (Global Surrogate) ---")

# Utilizziamo le predizioni della Rete Neurale sul test set come target per l'albero
global_surrogate = DecisionTreeClassifier(max_depth=4, random_state=42)
global_surrogate.fit(X_test, y_pred) # y_pred sono le etichette predette dal best_model

# Plot dell'albero decisionale globale
plt.figure(figsize=(20, 10))
plot_tree(global_surrogate, 
          feature_names=feature_names, 
          class_names=class_names, 
          filled=True, 
          rounded=True, 
          fontsize=10)
plt.title("Albero Decisionale Surrogato Globale (Approssimazione della Rete Neurale)")
plt.tight_layout()
plt.show()


--- 1. SPIEGAZIONE GLOBALE (Global Surrogate) ---


<Figure size 2000x1000 with 1 Axes>

In [23]:
# ==============================================================================
# [2. Instance Selection]
# Selezioniamo un'istanza in cui la Rete Neurale ha predetto una classe minoritaria
# (es. classe 3 = Dipendenza Severa, oppure classe 2).
# ==============================================================================
print("\n--- 2. SETUP SPIEGAZIONE LOCALE ---")

# Cerchiamo gli indici nel test set dove il modello ha predetto la classe 3 o 2
minority_class_indices = np.where(y_pred == 3)[0]

if len(minority_class_indices) > 0:
    target_idx = minority_class_indices[0]
else:
    # Fallback: prendiamo una riga a caso se non ci sono classi minoritarie predette
    target_idx = 0 

instance_scaled = X_test[target_idx].reshape(1, -1)
predicted_class = y_pred[target_idx]
true_class = y_test[target_idx]

print(f"Selezionata Istanza Indice: {target_idx}")
print(f"Classe Reale: {class_names[int(true_class)]}")
print(f"Classe Predetta dalla Rete: {class_names[predicted_class]}")


--- 2. SETUP SPIEGAZIONE LOCALE ---
Selezionata Istanza Indice: 26
Classe Reale: 3 (Severe)
Classe Predetta dalla Rete: 3 (Severe)


In [24]:
# ==============================================================================
# [3. SHAP Local & Global]
# Spiegazione tramite Feature Attribution usando i valori di Shapley
# ==============================================================================
print("\n--- 3. SPIEGAZIONE SHAP (Feature Attribution) ---")

# Inizializziamo l'Explainer di SHAP
background_data = shap.sample(X_train_resampled, 100, random_state=42)

def f_predict(x):
    return best_model.predict(x, verbose=0)

explainer = shap.KernelExplainer(f_predict, background_data)

# Calcoliamo gli SHAP values per la singola istanza
shap_values_instance = explainer.shap_values(instance_scaled, silent=True)

# GESTIONE COMPATIBILITA' VERSIONI SHAP
if isinstance(shap_values_instance, list):
    # SHAP < 0.40: lista di array, prendiamo l'array della classe predetta e la riga 0
    shap_vals = shap_values_instance[predicted_class][0]
else:
    # SHAP >= 0.40: array 3D (n_samples, n_features, n_classes)
    # Selezioniamo il sample 0, tutte le feature (:), e la classe predetta
    shap_vals = shap_values_instance[0, :, predicted_class]

expected_value = explainer.expected_value[predicted_class]

# Costruiamo l'oggetto Explanation per i plot più moderni di SHAP (come waterfall)
explanation = shap.Explanation(values=shap_vals,
                               base_values=expected_value,
                               data=instance_scaled[0],
                               feature_names=feature_names)

# Plot LOCALE (Waterfall Plot)
plt.figure()
shap.waterfall_plot(explanation, show=False)
plt.title(f"SHAP Waterfall Plot Locale (Previsione: {class_names[predicted_class]})")
plt.tight_layout()
plt.show()

# Plot GLOBALE (Summary Plot) su un sottoinsieme del Test Set per efficienza
test_subset = X_test[:150]
shap_values_global = explainer.shap_values(test_subset, silent=True)

# Assicuriamoci che il summary plot multiclasse riceva il formato giusto
if not isinstance(shap_values_global, list) and len(shap_values_global.shape) == 3:
    # Se è un array 3D, lo trasformiamo in una lista di array 2D per il summary_plot
    shap_values_global = [shap_values_global[:, :, i] for i in range(shap_values_global.shape[2])]

plt.figure()
shap.summary_plot(shap_values_global, test_subset, feature_names=feature_names, class_names=class_names, show=False)
plt.title("SHAP Summary Plot Globale sul Test Set")
plt.tight_layout()
plt.show()


--- 3. SPIEGAZIONE SHAP (Feature Attribution) ---


<Figure size 800x650 with 3 Axes>

<Figure size 800x950 with 1 Axes>

In [37]:
# ==============================================================================
# [4. LORE Rules & Counterfactuals]
# Implementazione basata sulla logica di LORE: generazione di vicinato sintetico,
# addestramento albero locale ed estrazione della Factual Rule e Counterfactuals.
# ==============================================================================
print("\n--- 4. SPIEGAZIONE LOCALE LORE (Factual & Counterfactual Rules) ---")

# Step 4.1: Generazione del vicinato sintetico tramite perturbazione
np.random.seed(42)
n_synthetic = 1500
noise = np.random.normal(0, 0.2, (n_synthetic, X_train_resampled.shape[1]))
neighborhood = instance_scaled + noise

# Step 4.2: Interrogazione della Black Box sul vicinato
neighborhood_preds = np.argmax(best_model.predict(neighborhood, verbose=0), axis=1)

# Step 4.3: Addestramento del Surrogato Locale (Albero Decisionale)
local_tree = DecisionTreeClassifier(max_depth=4, min_samples_leaf=5, random_state=42)
local_tree.fit(neighborhood, neighborhood_preds)

# Step 4.4: Estrazione della regola fattuale (Factual Rule)
node_indicator = local_tree.decision_path(instance_scaled)
leaf_id = local_tree.apply(instance_scaled)[0]
feature = local_tree.tree_.feature
threshold = local_tree.tree_.threshold

factual_rule = []
node_index = node_indicator.indices[node_indicator.indptr[0]:node_indicator.indptr[1]]

for node_id in node_index:
    if leaf_id == node_id:
        continue
        
    feat_idx = feature[node_id]
    scaled_threshold = threshold[node_id]
    
    # Creiamo un array fittizio (di zeri) per invertire la singola soglia
    dummy_array = np.zeros((1, X_train_resampled.shape[1]))
    dummy_array[0, feat_idx] = scaled_threshold
    
    # Calcoliamo il valore reale della soglia
    real_threshold = scaler.inverse_transform(dummy_array)[0, feat_idx]
    
    # Calcoliamo il valore reale dell'istanza
    real_instance_val = scaler.inverse_transform(instance_scaled)[0, feat_idx]
    
    if real_instance_val <= real_threshold:
        threshold_sign = "<="
    else:
        threshold_sign = ">"
    
    rule = f"{feature_names[feat_idx]} {threshold_sign} {real_threshold:.2f}"
    factual_rule.append(rule)

print("\n--- REGOLA FATTUALE (Factual Rule) ---")
print(f"Se:")
for r in factual_rule:
    print(f"  - {r}")
print(f"Allora la classe predetta è: {class_names[predicted_class]}")

# Step 4.5: Estrazione del Controfattuale (Counterfactual)
# Cerchiamo le foglie dell'albero locale che portano a una classe DIVERSA
leaf_indices = np.where(local_tree.tree_.children_left == _tree.TREE_LEAF)[0]
counterfactual_found = False

print("\n--- SPIEGAZIONE CONTROFATTUALE (Counterfactual) ---")
for leaf in leaf_indices:
    leaf_class = np.argmax(local_tree.tree_.value[leaf][0])
    if leaf_class != predicted_class:
        cf_candidates = neighborhood[local_tree.apply(neighborhood) == leaf]
        if len(cf_candidates) > 0:
            cf_instance_scaled = cf_candidates[0]
            
            # --- TRASFORMAZIONE IN VALORI REALI ---
            instance_real = scaler.inverse_transform(instance_scaled)[0]
            cf_instance_real = scaler.inverse_transform(cf_instance_scaled.reshape(1, -1))[0]
            
            print(f"Per ottenere la classe {class_names[leaf_class]} invece di {class_names[predicted_class]}, ")
            print("sarebbe necessario modificare le seguenti feature (valori reali):")
            
            # Troviamo le differenze sui valori SCALATI per capire quali feature sono cambiate significativamente (es. > 0.05 deviazioni standard)
            diffs_scaled = cf_instance_scaled - instance_scaled[0]
            significant_changes = np.abs(diffs_scaled) > 0.05
            
            for i, changed in enumerate(significant_changes):
                if changed:
                    # Usiamo i valori reali per decidere se è un aumento o una diminuzione
                    real_diff = cf_instance_real[i] - instance_real[i]
                    direction = "aumentare" if real_diff > 0 else "diminuire"
                    
                    # Stampiamo la spiegazione usando i numeri REALI
                    print(f"  - {direction} '{feature_names[i]}' da {instance_real[i]:.2f} a {cf_instance_real[i]:.2f}")
            
            counterfactual_found = True
            break # Mostriamo solo un controfattuale per chiarezza

if not counterfactual_found:
    print("Nessun controfattuale esplorabile trovato nel vicinato immediato (la decisione è molto solida in questa regione).")


--- 4. SPIEGAZIONE LOCALE LORE (Factual & Counterfactual Rules) ---

--- REGOLA FATTUALE (Factual Rule) ---
Se:
  - SDS-SDS_Total_T <= 58.85
  - BIA-BIA_ICW > 28.26
  - BIA-BIA_LST <= 64.91
  - Body_Proportion <= -0.04
Allora la classe predetta è: 3 (Severe)

--- SPIEGAZIONE CONTROFATTUALE (Counterfactual) ---
Per ottenere la classe 0 (None) invece di 3 (Severe), 
sarebbe necessario modificare le seguenti feature (valori reali):
  - diminuire 'Basic_Demos-Age' da 9.00 a 8.71
  - diminuire 'Basic_Demos-Sex' da 0.00 a -0.03
  - diminuire 'CGAS-CGAS_Score' da 65.00 a 57.45
  - diminuire 'Physical-HeartRate' da 81.00 a 79.16
  - diminuire 'Fitness_Endurance-Max_Stage' da 5.00 a 4.83
  - diminuire 'FGC-FGC_CU_Zone' da 0.00 a -0.07
  - aumentare 'FGC-FGC_PU_Zone' da 0.00 a 0.11
  - diminuire 'FGC-FGC_SRL_Zone' da 1.00 a 0.92
  - aumentare 'FGC-FGC_SRR_Zone' da 1.00 a 1.08
  - aumentare 'BIA-BIA_BMC' da 4.24 a 7.11
  - diminuire 'BIA-BIA_BMI' da 17.97 a 17.12
  - aumentare 'BIA-BIA_BMR' da 1